# Cell 1 — Setup

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

# -------------------------------------------------------------------
# INPUT EXISTING KNN METRICS POINT LAYER
# -------------------------------------------------------------------
in_gpkg = r"E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics.gpkg"
in_layer = "wb_buildings_extent_point_knn"

# -------------------------------------------------------------------
# BROAD KNN SUMMARY TABLE
# Adjust this path if your large-k CSV has a different name
# -------------------------------------------------------------------
knn_folder = r"E:\World Bank deliverbale 1\knn"
large_knn_csv = os.path.join(
    knn_folder,
    "large_knn",
    "building_knn_large_k_summary.csv"
)

# -------------------------------------------------------------------
# OUTPUT NEW GPKG
# -------------------------------------------------------------------
out_gpkg = os.path.join(
    knn_folder,
    "buildings_points_knn_metrics_broad.gpkg"
)

out_layer = "wb_buildings_extent_point_knn_broad"

print("Input GPKG:", in_gpkg)
print("Input layer:", in_layer)
print("Large KNN CSV:", large_knn_csv)
print("Output GPKG:", out_gpkg)
print("Output layer:", out_layer)

Input GPKG: E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics.gpkg
Input layer: wb_buildings_extent_point_knn
Large KNN CSV: E:\World Bank deliverbale 1\knn\large_knn\building_knn_large_k_summary.csv
Output GPKG: E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics_broad.gpkg
Output layer: wb_buildings_extent_point_knn_broad


# Cell 2 — Confirm layer names

In [2]:
print("Layers in existing metrics GPKG:")
print(pyogrio.list_layers(in_gpkg))

Layers in existing metrics GPKG:
[['wb_buildings_extent_point_knn' 'Point']]


# Cell 3 — Read existing point layer

In [3]:
gdf = pyogrio.read_dataframe(in_gpkg, layer=in_layer)

print("Rows:", f"{len(gdf):,}")
print("Columns:")
print(gdf.columns.tolist())
print("CRS:", gdf.crs)

gdf.head()

Rows: 578,379
Columns:
['type', 'building', 'construction', 'area_m_utm', 'ORIG_FID', 'building_id', 'd5_near', 'd80_near', 'r80_5', 'd5_q4', 'd5_q5', 'd80_q4', 'd80_q5', 'r80_5_q4', 'r80_5_q5', 'geometry']
CRS: EPSG:32636


,type,building,construction,area_m_utm,ORIG_FID,building_id,d5_near,d80_near,r80_5,d5_q4,d5_q5,d80_q4,d80_q5,r80_5_q4,r80_5_q5,geometry
0,multipolygon,yes,None,297.854874,1,1,24.751932,139.705179,5.644213,4,4,4,5,4,5,POINT (344402.388 536236.527)
1,multipolygon,yes,None,134.675182,2,2,13.947397,85.227719,6.110654,2,2,3,4,4,5,POINT (343156.5 537298.568)
2,multipolygon,yes,None,254.245943,3,3,16.169194,64.679283,4.000155,2,3,1,2,2,2,POINT (343313.65 538071.93)
3,multipolygon,yes,None,331.438943,4,4,34.483712,113.930039,3.303880,4,5,4,5,1,1,POINT (346336.899 536569.205)
4,multipolygon,yes,None,431.678814,5,5,37.518905,139.469801,3.717321,4,5,4,5,1,2,POINT (343792.458 541426.962)


# Cell 4 — Read broad KNN summary table

In [4]:
large_knn = pd.read_csv(large_knn_csv)

print("Rows:", f"{len(large_knn):,}")
print("Columns:")
print(large_knn.columns.tolist())

large_knn.head()

Rows: 578,379
Columns:
['building_id', 'd200_near', 'd500_near', 'd1000_near', 'd2000_near', 'd4000_near', 'd8000_near']


,building_id,d200_near,d500_near,d1000_near,d2000_near,d4000_near,d8000_near
0,1,201.281452,298.442495,423.718285,546.447837,684.379194,923.563457
1,2,131.833827,208.683046,298.791597,422.931166,590.887785,823.640003
2,3,111.330332,180.006642,250.973065,363.245106,524.118767,771.978276
3,4,181.452948,267.846031,371.916636,491.732728,651.504992,1061.270135
4,5,190.199776,258.271489,325.549845,443.715436,652.874970,983.048727


# Cell 5 — Prepare fields for join

In [5]:
# Fields to add to the point layer
broad_cols = [
    "building_id",
    "d500_near",
    "d1000_near",
    "d2000_near",
    "d4000_near",
    "d8000_near",
]

missing = [c for c in broad_cols if c not in large_knn.columns]
if missing:
    raise ValueError(f"Missing expected columns in large KNN table: {missing}")

large_keep = large_knn[broad_cols].copy()

# Clean join keys
large_keep["building_id"] = pd.to_numeric(
    large_keep["building_id"],
    errors="coerce"
).astype("Int64")

if "ORIG_FID" not in gdf.columns:
    raise ValueError(f"ORIG_FID not found in point layer. Available columns: {gdf.columns.tolist()}")

gdf["ORIG_FID"] = pd.to_numeric(
    gdf["ORIG_FID"],
    errors="coerce"
).astype("Int64")

print("Unique ORIG_FID in points:", f"{gdf['ORIG_FID'].nunique():,}")
print("Unique building_id in large KNN:", f"{large_keep['building_id'].nunique():,}")

Unique ORIG_FID in points: 578,379
Unique building_id in large KNN: 578,379


# Cell 6 — Join broad-distance fields to the point layer

In [7]:
# Drop these fields from gdf if they already exist, to avoid _x/_y suffixes
fields_to_replace = [
    "d500_near",
    "d1000_near",
    "d2000_near",
    "d4000_near",
    "d8000_near",
]

existing = [c for c in fields_to_replace if c in gdf.columns]
if existing:
    print("Dropping existing broad fields before join:", existing)
    gdf = gdf.drop(columns=existing)

gdf_broad = gdf.merge(
    large_keep,
    left_on="ORIG_FID",
    right_on="building_id",
    how="left",
    validate="one_to_one"
)

print("Original rows:", f"{len(gdf):,}")
print("Joined rows:", f"{len(gdf_broad):,}")

matched = gdf_broad["d500_near"].notna().sum()
unmatched = len(gdf_broad) - matched

print("Matched:", f"{matched:,}")
print("Unmatched:", f"{unmatched:,}")
print("Match rate:", f"{matched / len(gdf_broad):.2%}")

gdf_broad[
    [
        "ORIG_FID",
        "d5_near",
        "d80_near",
        "d500_near",
        "d1000_near",
        "d2000_near",
        "d4000_near",
        "d8000_near",
    ]
].head()

Original rows: 578,379
Joined rows: 578,379
Matched: 578,379
Unmatched: 0
Match rate: 100.00%


,ORIG_FID,d5_near,d80_near,d500_near,d1000_near,d2000_near,d4000_near,d8000_near
0,1,24.751932,139.705179,298.442495,423.718285,546.447837,684.379194,923.563457
1,2,13.947397,85.227719,208.683046,298.791597,422.931166,590.887785,823.640003
2,3,16.169194,64.679283,180.006642,250.973065,363.245106,524.118767,771.978276
3,4,34.483712,113.930039,267.846031,371.916636,491.732728,651.504992,1061.270135
4,5,37.518905,139.469801,258.271489,325.549845,443.715436,652.874970,983.048727


# Cell 7 — Create ratio fields for later mapping

In [8]:
# Broad-to-local ratio fields
gdf_broad["r500_80"] = gdf_broad["d500_near"] / gdf_broad["d80_near"]
gdf_broad["r1000_80"] = gdf_broad["d1000_near"] / gdf_broad["d80_near"]
gdf_broad["r2000_80"] = gdf_broad["d2000_near"] / gdf_broad["d80_near"]
gdf_broad["r4000_80"] = gdf_broad["d4000_near"] / gdf_broad["d80_near"]
gdf_broad["r8000_80"] = gdf_broad["d8000_near"] / gdf_broad["d80_near"]

# Clean infinite values
ratio_cols = [
    "r500_80",
    "r1000_80",
    "r2000_80",
    "r4000_80",
    "r8000_80",
]

for col in ratio_cols:
    gdf_broad[col] = gdf_broad[col].replace([np.inf, -np.inf], np.nan)

gdf_broad[
    [
        "d80_near",
        "d500_near",
        "d1000_near",
        "d2000_near",
        "d4000_near",
        "d8000_near",
        "r8000_80",
    ]
].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

,d80_near,d500_near,d1000_near,d2000_near,d4000_near,d8000_near,r8000_80
count,578379.000000,578379.000000,578379.000000,578379.000000,578379.000000,578379.000000,578379.000000
mean,92.707942,240.307203,348.321348,501.294662,716.859555,1021.817641,11.768811
std,58.166055,146.890145,224.471190,313.851742,406.662425,514.512403,3.870307
min,23.848991,68.401856,112.642138,208.989321,316.475520,549.844548,2.058529
25%,64.878347,169.882265,244.467075,352.337409,507.131811,732.631992,9.836958
50%,76.872505,197.885588,282.703160,404.029708,581.617653,843.060662,11.060760
75%,100.963016,259.134692,372.262331,534.040039,769.786181,1112.065276,12.711344
90%,141.857113,365.327013,528.276493,768.024664,1105.217704,1578.833429,15.507594
95%,178.048361,460.317240,662.959114,964.543012,1372.066823,1895.073238,18.201800
99%,328.033336,888.880297,1330.137146,1851.937411,2482.016670,3178.354097,24.870336


# Cell 8 — Optional: create quartile fields for mapping

In [9]:
def add_quantile_class(df, value_col, q=4, out_col=None):
    if out_col is None:
        out_col = f"{value_col}_q{q}"

    df[out_col] = pd.Series([pd.NA] * len(df), dtype="Int64")
    valid = df[value_col].replace([np.inf, -np.inf], np.nan).notna()

    df.loc[valid, out_col] = (
        pd.qcut(
            df.loc[valid, value_col],
            q=q,
            labels=False,
            duplicates="drop"
        ) + 1
    ).astype("Int64")

    return df


map_cols = [
    "d500_near",
    "d1000_near",
    "d2000_near",
    "d4000_near",
    "d8000_near",
    "r8000_80",
]

for col in map_cols:
    gdf_broad = add_quantile_class(gdf_broad, col, q=4, out_col=f"{col}_q4")

print([c for c in gdf_broad.columns if c.endswith("_q4")])

['d5_q4', 'd80_q4', 'r80_5_q4', 'd500_near_q4', 'd1000_near_q4', 'd2000_near_q4', 'd4000_near_q4', 'd8000_near_q4', 'r8000_80_q4']


# Cell 9 — Export new GeoPackage layer

In [10]:
# Remove old output if it exists
if os.path.exists(out_gpkg):
    os.remove(out_gpkg)

# Optional: keep a manageable set of fields
keep_cols = [
    "type",
    "building",
    "construction",
    "area_m_utm",
    "ORIG_FID",
    "building_id",
    "d5_near",
    "d80_near",
    "r80_5",
    "d500_near",
    "d1000_near",
    "d2000_near",
    "d4000_near",
    "d8000_near",
    "r500_80",
    "r1000_80",
    "r2000_80",
    "r4000_80",
    "r8000_80",
    "d500_near_q4",
    "d1000_near_q4",
    "d2000_near_q4",
    "d4000_near_q4",
    "d8000_near_q4",
    "r8000_80_q4",
    "geometry",
]

keep_cols = [c for c in keep_cols if c in gdf_broad.columns]

export_gdf = gdf_broad[keep_cols].copy()

export_gdf.to_file(out_gpkg, layer=out_layer, driver="GPKG")

print("Exported:")
print(out_gpkg)
print("Layer:", out_layer)
print("Rows:", f"{len(export_gdf):,}")
print("Columns:")
print(export_gdf.columns.tolist())

Exported:
E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics_broad.gpkg
Layer: wb_buildings_extent_point_knn_broad
Rows: 578,379
Columns:
['type', 'building', 'construction', 'area_m_utm', 'ORIG_FID', 'd5_near', 'd80_near', 'r80_5', 'd500_near', 'd1000_near', 'd2000_near', 'd4000_near', 'd8000_near', 'r500_80', 'r1000_80', 'r2000_80', 'r4000_80', 'r8000_80', 'd500_near_q4', 'd1000_near_q4', 'd2000_near_q4', 'd4000_near_q4', 'd8000_near_q4', 'r8000_80_q4', 'geometry']
